# Data Cleaning Pipeline

This notebook creates a modeling-ready version of the OPTN kidney waitlist analytic dataset.

The original dataset will be preserved as `df_raw`.

All cleaning steps will be performed on a separate copy called `df_clean`.

Main cleaning goals:

- preserve the original data
- correct data types
- identify invalid values
- handle missingness appropriately
- prevent target leakage
- preserve meaningful structural missingness
- prepare safe listing-time features for modeling
- document every cleaning decision

In [ ]:
# ============================================
# STEP 1 — Load and protect the raw dataset
# ============================================

import pandas as pd
import numpy as np

dataset_url = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/"
    "Healthcare-1A-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation/"
    "main/data/kidney_waitlist_analytic.csv.gz"
)

# Load original dataset
df_raw = pd.read_csv(
    dataset_url,
    compression="gzip",
    low_memory=False
)

# Create a separate working copy for cleaning
df_clean = df_raw.copy(deep=True)

print("Dataset loaded successfully!")
print("Raw shape:", df_raw.shape)
print("Cleaning copy shape:", df_clean.shape)

print("\nSame number of rows:",
      len(df_raw) == len(df_clean))

print("Same number of columns:",
      df_raw.shape[1] == df_clean.shape[1])

print("\nFirst 5 columns:")
print(df_clean.columns[:5].tolist())

Dataset loaded successfully!
Raw shape: (494862, 38)
Cleaning copy shape: (494862, 38)

Same number of rows: True
Same number of columns: True

First 5 columns:
['ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR']


## Step 2 — Date Conversion and Validation

The date columns are converted to proper datetime format.

We then check for invalid date values and impossible timelines, especially cases where `END_DATE` occurs before `INIT_DATE`.

The original rows are not deleted in this step. Invalid timelines are only flagged for later handling.

In [ ]:
# ============================================
# STEP 2 — Convert and validate date columns
# ============================================

date_columns = [
    "INIT_DATE",
    "END_DATE",
    "DIALYSIS_DATE",
    "TX_DATE",
    "COMPOSITE_DEATH_DATE"
]

# Convert text dates into pandas datetime values
for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

print("DATE CONVERSION COMPLETE")

# Check missing/invalid dates after conversion
for col in date_columns:
    print(
        f"{col}:",
        df_clean[col].isna().sum(),
        "missing/invalid"
    )


# Calculate timeline difference
df_clean["calculated_days_to_end"] = (
    df_clean["END_DATE"] - df_clean["INIT_DATE"]
).dt.days


# Flag impossible negative timelines
df_clean["invalid_date_order"] = (
    df_clean["calculated_days_to_end"] < 0
)


print("\nTIMELINE VALIDATION")
print(
    "END_DATE before INIT_DATE:",
    df_clean["invalid_date_order"].sum()
)

print(
    "Valid/non-negative timelines:",
    (~df_clean["invalid_date_order"]).sum()
)

print(
    "\nMinimum calculated duration:",
    df_clean["calculated_days_to_end"].min()
)

print(
    "Maximum calculated duration:",
    df_clean["calculated_days_to_end"].max()
)

DATE CONVERSION COMPLETE
INIT_DATE: 0 missing/invalid
END_DATE: 0 missing/invalid
DIALYSIS_DATE: 134245 missing/invalid
TX_DATE: 264107 missing/invalid
COMPOSITE_DEATH_DATE: 398512 missing/invalid

TIMELINE VALIDATION
END_DATE before INIT_DATE: 59
Valid/non-negative timelines: 494803

Minimum calculated duration: -4810
Maximum calculated duration: 4200


## Step 3 — BMI Quality Flags

EDA identified extreme and implausible values in `BMI_TCR`.

Before modifying BMI, values are grouped into quality-check categories so that suspicious observations can be reviewed and handled consistently.

No BMI values are replaced or deleted in this step.

In [ ]:
# ============================================
# STEP 3 — Flag suspicious BMI values
# ============================================

def bmi_quality_flag(x):

    if pd.isna(x):
        return "Missing"

    elif x <= 10:
        return "Very low (<=10)"

    elif x <= 60:
        return "Main observed range (10-60)"

    elif x <= 100:
        return "Very high (60-100)"

    else:
        return "Extreme (>100)"


df_clean["BMI_quality_flag"] = (
    df_clean["BMI_TCR"].apply(bmi_quality_flag)
)


bmi_flag_summary = (
    df_clean["BMI_quality_flag"]
    .value_counts()
    .rename_axis("BMI_Quality_Group")
    .reset_index(name="Count")
)

bmi_flag_summary["Percent"] = (
    bmi_flag_summary["Count"] / len(df_clean) * 100
).round(3)

print(bmi_flag_summary.to_string(index=False))


print("\nBMI ORIGINAL RANGE")
print("Minimum BMI:", df_clean["BMI_TCR"].min())
print("Maximum BMI:", df_clean["BMI_TCR"].max())


print("\nEXTREME CHECK")
print("BMI <= 10:", (df_clean["BMI_TCR"] <= 10).sum())
print("BMI > 60:", (df_clean["BMI_TCR"] > 60).sum())
print("BMI > 100:", (df_clean["BMI_TCR"] > 100).sum())
print("BMI > 1000:", (df_clean["BMI_TCR"] > 1000).sum())

          BMI_Quality_Group  Count  Percent
Main observed range (10-60) 492590   99.541
                    Missing   1558    0.315
             Extreme (>100)    263    0.053
            Very low (<=10)    240    0.048
         Very high (60-100)    211    0.043

BMI ORIGINAL RANGE
Minimum BMI: 0.18
Maximum BMI: 430226.67

EXTREME CHECK
BMI <= 10: 240
BMI > 60: 474
BMI > 100: 263
BMI > 1000: 124


## Step 4 — Create Clean BMI Variable

The original BMI column is preserved.

A separate cleaned BMI feature is created for modeling.

Values at or below 10 and values above 100 are treated as invalid/suspicious and converted to missing values.

Very high values between 60 and 100 are retained for now but remain flagged for review.

This rule can be revised if the project advisor or OPTN documentation provides a more appropriate clinical range.

In [ ]:
# ============================================
# STEP 4 — Create cleaned BMI feature
# ============================================

# Start with an exact copy of the original BMI
df_clean["BMI_TCR_clean"] = df_clean["BMI_TCR"].copy()

# Flag values we will treat as invalid for modeling
bmi_invalid_mask = (
    (df_clean["BMI_TCR_clean"] <= 10) |
    (df_clean["BMI_TCR_clean"] > 100)
)

print("BMI values marked invalid:", bmi_invalid_mask.sum())

# Replace only those BMI values with missing
df_clean.loc[
    bmi_invalid_mask,
    "BMI_TCR_clean"
] = np.nan


print("\nBEFORE CLEANING")
print("Missing BMI:", df_clean["BMI_TCR"].isna().sum())
print("Minimum:", df_clean["BMI_TCR"].min())
print("Maximum:", df_clean["BMI_TCR"].max())


print("\nAFTER CLEANING")
print("Missing BMI:", df_clean["BMI_TCR_clean"].isna().sum())
print("Minimum:", df_clean["BMI_TCR_clean"].min())
print("Maximum:", df_clean["BMI_TCR_clean"].max())


print("\nOriginal BMI column unchanged:")
print(
    df_clean["BMI_TCR"].equals(df_raw["BMI_TCR"])
)

BMI values marked invalid: 503

BEFORE CLEANING
Missing BMI: 1558
Minimum: 0.18
Maximum: 430226.67

AFTER CLEANING
Missing BMI: 2061
Minimum: 10.03
Maximum: 97.65

Original BMI column unchanged:
True


## Step 5 — Functional Status Quality Check

EDA identified one unusual `FUNC_STAT_TCR` value that does not match the main observed coding schemes.

Before changing anything, all observed functional-status codes are reviewed.

No values are modified in this step.

In [ ]:
# ============================================
# STEP 5 — Inspect functional status codes
# ============================================

func_status_counts = (
    df_clean["FUNC_STAT_TCR"]
    .value_counts(dropna=False)
    .sort_index()
)

print("FUNCTIONAL STATUS VALUE COUNTS")
print(func_status_counts)

print("\nMissing functional status:",
      df_clean["FUNC_STAT_TCR"].isna().sum())

print(
    "Records with FUNC_STAT_TCR = 1:",
    (df_clean["FUNC_STAT_TCR"] == 1).sum()
)

print(
    "Records with FUNC_STAT_TCR = 998:",
    (df_clean["FUNC_STAT_TCR"] == 998).sum()
)

FUNCTIONAL STATUS VALUE COUNTS
FUNC_STAT_TCR
1.0            1
996.0        121
998.0       8700
2010.0       371
2020.0      2936
2030.0      1905
2040.0      8364
2050.0     23923
2060.0     36967
2070.0    111493
2080.0    139478
2090.0    109316
2100.0     35412
4010.0        43
4020.0        23
4030.0        89
4040.0       187
4050.0       157
4060.0       505
4070.0      1029
4080.0      2812
4090.0      2413
4100.0      4642
NaN         3975
Name: count, dtype: int64

Missing functional status: 3975
Records with FUNC_STAT_TCR = 1: 1
Records with FUNC_STAT_TCR = 998: 8700


## Step 6 — Clean Functional Status

`FUNC_STAT_TCR` is a coded categorical variable.

One record contains the value `1`, which does not match the observed functional-status coding schemes and is treated as invalid.

Special codes such as `996` and `998` are preserved because their exact meanings should be confirmed from the official OPTN data dictionary before recoding.

The original column is preserved.

In [ ]:
# ============================================
# STEP 6 — Clean functional status
# ============================================

# Preserve original column
df_clean["FUNC_STAT_TCR_clean"] = df_clean["FUNC_STAT_TCR"].copy()

# Flag clearly invalid value
df_clean["FUNC_STAT_invalid_flag"] = (
    df_clean["FUNC_STAT_TCR_clean"] == 1
)

print(
    "Invalid functional status records:",
    df_clean["FUNC_STAT_invalid_flag"].sum()
)

# Replace only clearly invalid value with missing
df_clean.loc[
    df_clean["FUNC_STAT_invalid_flag"],
    "FUNC_STAT_TCR_clean"
] = np.nan


print("\nBEFORE CLEANING")
print(
    "Missing:",
    df_clean["FUNC_STAT_TCR"].isna().sum()
)

print("\nAFTER CLEANING")
print(
    "Missing:",
    df_clean["FUNC_STAT_TCR_clean"].isna().sum()
)

print(
    "Remaining value = 1:",
    (df_clean["FUNC_STAT_TCR_clean"] == 1).sum()
)

print(
    "Code 996 preserved:",
    (df_clean["FUNC_STAT_TCR_clean"] == 996).sum()
)

print(
    "Code 998 preserved:",
    (df_clean["FUNC_STAT_TCR_clean"] == 998).sum()
)

print(
    "\nOriginal column unchanged:",
    df_clean["FUNC_STAT_TCR"].equals(df_raw["FUNC_STAT_TCR"])
)

Invalid functional status records: 1

BEFORE CLEANING
Missing: 3975

AFTER CLEANING
Missing: 3976
Remaining value = 1: 0
Code 996 preserved: 121
Code 998 preserved: 8700

Original column unchanged: True


## Step 7 — Initial CPRA Validation and Missingness

`INIT_CPRA` is an important listing-time feature, but EDA showed that its missingness changes sharply over time.

Values are first checked for valid range and temporal missingness.

A missingness indicator is created, but missing CPRA values are not imputed yet.

Imputation will be handled later using training data only to avoid data leakage.

In [ ]:
# ============================================
# STEP 7 — Validate Initial CPRA
# ============================================

# Listing year from INIT_DATE
df_clean["listing_year"] = df_clean["INIT_DATE"].dt.year

# Missingness flag
df_clean["INIT_CPRA_missing"] = (
    df_clean["INIT_CPRA"].isna()
).astype(int)


print("INITIAL CPRA RANGE")
print("Minimum:", df_clean["INIT_CPRA"].min())
print("Maximum:", df_clean["INIT_CPRA"].max())


# Check values outside expected 0-100 range
cpra_invalid_mask = (
    (df_clean["INIT_CPRA"] < 0) |
    (df_clean["INIT_CPRA"] > 100)
)

print(
    "\nValues outside 0-100:",
    cpra_invalid_mask.sum()
)


print("\nOVERALL MISSINGNESS")
print(
    "Missing INIT_CPRA:",
    df_clean["INIT_CPRA"].isna().sum()
)

print(
    "Missing percent:",
    round(
        df_clean["INIT_CPRA"].isna().mean() * 100,
        2
    ),
    "%"
)


# Missingness by listing year
cpra_by_year = (
    df_clean
    .groupby("listing_year")
    .agg(
        total_records=("INIT_CPRA", "size"),
        missing_cpra=("INIT_CPRA_missing", "sum")
    )
)

cpra_by_year["missing_percent"] = (
    cpra_by_year["missing_cpra"]
    / cpra_by_year["total_records"]
    * 100
).round(2)

print("\nCPRA MISSINGNESS BY YEAR")
print(cpra_by_year)

INITIAL CPRA RANGE
Minimum: 0.0
Maximum: 100.0

Values outside 0-100: 0

OVERALL MISSINGNESS
Missing INIT_CPRA: 142741
Missing percent: 28.84 %

CPRA MISSINGNESS BY YEAR
              total_records  missing_cpra  missing_percent
listing_year                                              
2015                  36599             0             0.00
2016                  36956             0             0.00
2017                  37209             0             0.00
2018                  40564             0             0.00
2019                  43223             0             0.00
2020                  37825             0             0.00
2021                  42130             0             0.00
2022                  44442             0             0.00
2023                  47003         37380            79.53
2024                  49635         41129            82.86
2025                  53231         43199            81.15
2026                  26045         21033            80.76


## Step 8 — Dialysis Date Consistency Check

`DIALYSIS_DATE` is structurally missing for patients who are not on dialysis.

This step checks whether dialysis dates are logically consistent with dialysis status and listing date.

No dialysis values are modified yet.

In [ ]:
# ============================================
# STEP 8 — Validate dialysis date consistency
# ============================================

print("DIALYSIS STATUS COUNTS")
print(df_clean["ON_DIALYSIS"].value_counts(dropna=False))


# Check missing dialysis date by dialysis status
dialysis_missing_check = (
    df_clean
    .groupby("ON_DIALYSIS")
    .agg(
        total_records=("DIALYSIS_DATE", "size"),
        missing_dialysis_date=("DIALYSIS_DATE", lambda x: x.isna().sum())
    )
)

dialysis_missing_check["missing_percent"] = (
    dialysis_missing_check["missing_dialysis_date"]
    / dialysis_missing_check["total_records"]
    * 100
).round(2)

print("\nDIALYSIS DATE MISSINGNESS BY STATUS")
print(dialysis_missing_check)


# Compare dialysis date with listing date
on_dialysis = df_clean["ON_DIALYSIS"] == "Y"

dialysis_before = (
    on_dialysis &
    (df_clean["DIALYSIS_DATE"] < df_clean["INIT_DATE"])
)

dialysis_same_day = (
    on_dialysis &
    (df_clean["DIALYSIS_DATE"] == df_clean["INIT_DATE"])
)

dialysis_after = (
    on_dialysis &
    (df_clean["DIALYSIS_DATE"] > df_clean["INIT_DATE"])
)


print("\nDIALYSIS DATE VS LISTING DATE")
print("Started before listing:", dialysis_before.sum())
print("Started on listing date:", dialysis_same_day.sum())
print("Started after listing:", dialysis_after.sum())


# How far after listing?
days_after_listing = (
    df_clean.loc[dialysis_after, "DIALYSIS_DATE"]
    - df_clean.loc[dialysis_after, "INIT_DATE"]
).dt.days

print("\nFOR DIALYSIS DATES AFTER LISTING")
print("Count:", len(days_after_listing))

if len(days_after_listing) > 0:
    print("Median days after listing:", days_after_listing.median())
    print("Maximum days after listing:", days_after_listing.max())

DIALYSIS STATUS COUNTS
ON_DIALYSIS
Y    360617
N    134245
Name: count, dtype: int64

DIALYSIS DATE MISSINGNESS BY STATUS
             total_records  missing_dialysis_date  missing_percent
ON_DIALYSIS                                                       
N                   134245                 134245            100.0
Y                   360617                      0              0.0

DIALYSIS DATE VS LISTING DATE
Started before listing: 318009
Started on listing date: 236
Started after listing: 42372

FOR DIALYSIS DATES AFTER LISTING
Count: 42372
Median days after listing: 250.0
Maximum days after listing: 3792


## Step 9 — A2/A2B Eligibility Missingness Check

`A2A2B_ELIGIBILITY` has very high overall missingness, but EDA showed that the field is mainly relevant for blood type B candidates.

This step checks whether the missingness is structural and whether the observed values are valid.

No values are changed yet.

In [ ]:
# ============================================
# STEP 9 — Validate A2/A2B eligibility
# ============================================

print("A2A2B ELIGIBILITY VALUE COUNTS")
print(
    df_clean["A2A2B_ELIGIBILITY"]
    .value_counts(dropna=False)
)

print("\nBY BLOOD TYPE")

a2_by_abo = (
    df_clean
    .groupby("ABO")
    .agg(
        total_records=("A2A2B_ELIGIBILITY", "size"),
        nonmissing=("A2A2B_ELIGIBILITY", lambda x: x.notna().sum()),
        missing=("A2A2B_ELIGIBILITY", lambda x: x.isna().sum())
    )
)

a2_by_abo["nonmissing_percent"] = (
    a2_by_abo["nonmissing"]
    / a2_by_abo["total_records"]
    * 100
).round(2)

print(a2_by_abo)


# Check whether any non-B patient has this field populated
non_b_with_value = (
    (df_clean["ABO"] != "B") &
    (df_clean["A2A2B_ELIGIBILITY"].notna())
)

print(
    "\nNon-B patients with A2A2B value:",
    non_b_with_value.sum()
)


# Check unexpected values
valid_a2_values = [0, 1]

unexpected_a2 = (
    df_clean["A2A2B_ELIGIBILITY"].notna() &
    ~df_clean["A2A2B_ELIGIBILITY"].isin(valid_a2_values)
)

print(
    "Unexpected values outside 0/1:",
    unexpected_a2.sum()
)

A2A2B ELIGIBILITY VALUE COUNTS
A2A2B_ELIGIBILITY
NaN    456604
0.0     27460
1.0     10798
Name: count, dtype: int64

BY BLOOD TYPE
     total_records  nonmissing  missing  nonmissing_percent
ABO                                                        
A           155594           0   155594                0.00
A1            2398           0     2398                0.00
A1B            226           0      226                0.00
A2             365           0      365                0.00
A2B             75           0       75                0.00
AB           18475           0    18475                0.00
B            74036       38258    35778               51.67
O           243693           0   243693                0.00

Non-B patients with A2A2B value: 0
Unexpected values outside 0/1: 0


## Step 10 — Structure A2/A2B Eligibility

`A2A2B_ELIGIBILITY` is only populated for blood type B candidates.

Missing values for non-B candidates are structural and should not be treated as ordinary missing data.

A cleaned categorical feature is created to distinguish:

- non-B / not applicable
- B with recorded value 0
- B with recorded value 1
- B with missing eligibility

In [ ]:
# ============================================
# STEP 10 — Structure A2/A2B eligibility
# ============================================

def create_a2a2b_status(row):

    if row["ABO"] != "B":
        return "Not applicable (non-B)"

    if pd.isna(row["A2A2B_ELIGIBILITY"]):
        return "B - missing"

    if row["A2A2B_ELIGIBILITY"] == 0:
        return "B - recorded 0"

    if row["A2A2B_ELIGIBILITY"] == 1:
        return "B - recorded 1"

    return "Unexpected"


df_clean["A2A2B_status"] = df_clean.apply(
    create_a2a2b_status,
    axis=1
)

print("A2/A2B CLEAN STATUS")
print(
    df_clean["A2A2B_status"]
    .value_counts()
)

print("\nUnexpected status count:")
print(
    (df_clean["A2A2B_status"] == "Unexpected").sum()
)

A2/A2B CLEAN STATUS
A2A2B_status
Not applicable (non-B)    420826
B - missing                35778
B - recorded 0             27460
B - recorded 1             10798
Name: count, dtype: int64

Unexpected status count:
0


## Step 11 — Define Modeling-Safe and Leakage Columns

Prediction is made from the time of waitlist registration.

Therefore, model features should only contain information available at or near `INIT_DATE`.

Columns containing end-of-waitlist information, transplant information, outcomes, or future timing are preserved in the cleaned dataset for analysis but excluded from baseline model features.

In [ ]:
# ============================================
# STEP 11 — Define feature roles and leakage
# ============================================

# Main cleaned listing-time features
safe_listing_features = [
    "INIT_AGE",
    "GENDER",
    "ETHCAT",
    "ABO",
    "INIT_CPRA",
    "INIT_CPRA_missing",
    "ON_DIALYSIS",
    "BMI_TCR_clean",
    "FUNC_STAT_TCR_clean",
    "INIT_STAT",
    "REGION",
    "A2A2B_status"
]


# IDs — useful for tracking/splitting, but not model predictors
id_columns = [
    "PT_CODE",
    "WL_ID_CODE",
    "TRR_ID_CODE",
    "DONOR_ID"
]


# Target / outcome columns
target_columns = [
    "outcome",
    "event_adverse",
    "event_transplant",
    "censored",
    "days_to_event"
]


# Future information / leakage columns
leakage_columns = [
    "END_DATE",
    "END_CPRA",
    "END_STAT",
    "REM_CD",
    "TX_DATE",
    "COMPOSITE_DEATH_DATE",
    "DAYSWAIT_CHRON"
]


print("SAFE LISTING-TIME FEATURES")
for col in safe_listing_features:
    print("✓", col)


print("\nID COLUMNS — NOT MODEL FEATURES")
for col in id_columns:
    print("•", col)


print("\nTARGET / OUTCOME COLUMNS")
for col in target_columns:
    print("•", col)


print("\nFUTURE / LEAKAGE COLUMNS")
for col in leakage_columns:
    print("✗", col)


# Check that all listed columns actually exist
all_role_columns = (
    safe_listing_features
    + id_columns
    + target_columns
    + leakage_columns
)

missing_role_columns = [
    col for col in all_role_columns
    if col not in df_clean.columns
]

print("\nCOLUMN VALIDATION")
print("Missing listed columns:", missing_role_columns)

print(
    "Number of safe baseline features:",
    len(safe_listing_features)
)

SAFE LISTING-TIME FEATURES
✓ INIT_AGE
✓ GENDER
✓ ETHCAT
✓ ABO
✓ INIT_CPRA
✓ INIT_CPRA_missing
✓ ON_DIALYSIS
✓ BMI_TCR_clean
✓ FUNC_STAT_TCR_clean
✓ INIT_STAT
✓ REGION
✓ A2A2B_status

ID COLUMNS — NOT MODEL FEATURES
• PT_CODE
• WL_ID_CODE
• TRR_ID_CODE
• DONOR_ID

TARGET / OUTCOME COLUMNS
• outcome
• event_adverse
• event_transplant
• censored
• days_to_event

FUTURE / LEAKAGE COLUMNS
✗ END_DATE
✗ END_CPRA
✗ END_STAT
✗ REM_CD
✗ TX_DATE
✗ COMPOSITE_DEATH_DATE
✗ DAYSWAIT_CHRON

COLUMN VALIDATION
Missing listed columns: []
Number of safe baseline features: 12


## Step 12 — Validate Baseline Categorical Features

The main categorical listing-time variables are inspected for:

- missing values
- unexpected categories
- inconsistent formatting
- unusual codes

No categorical values are modified in this step.

In [ ]:
# ============================================
# STEP 12 — Validate categorical variables
# ============================================

categorical_baseline = [
    "GENDER",
    "ABO",
    "ETHCAT",
    "ON_DIALYSIS",
    "INIT_STAT",
    "REGION"
]

for col in categorical_baseline:

    print("\n" + "=" * 55)
    print(col)
    print("=" * 55)

    print("Missing:", df_clean[col].isna().sum())
    print("Unique values:", df_clean[col].nunique(dropna=True))

    print("\nValue counts:")
    print(
        df_clean[col]
        .value_counts(dropna=False)
        .sort_index()
    )


# Check blank strings in text columns
text_columns = ["GENDER", "ABO", "ON_DIALYSIS"]

print("\n" + "=" * 55)
print("BLANK STRING CHECK")
print("=" * 55)

for col in text_columns:

    blank_count = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"{col}: {blank_count} blank values")


GENDER
Missing: 0
Unique values: 2

Value counts:
GENDER
F    188454
M    306408
Name: count, dtype: int64

ABO
Missing: 0
Unique values: 8

Value counts:
ABO
A      155594
A1       2398
A1B       226
A2        365
A2B        75
AB      18475
B       74036
O      243693
Name: count, dtype: int64

ETHCAT
Missing: 0
Unique values: 8

Value counts:
ETHCAT
1      202625
2      144926
4       97884
5       36995
6        3949
7        2522
9        4182
998      1779
Name: count, dtype: int64

ON_DIALYSIS
Missing: 0
Unique values: 2

Value counts:
ON_DIALYSIS
N    134245
Y    360617
Name: count, dtype: int64

INIT_STAT
Missing: 0
Unique values: 5

Value counts:
INIT_STAT
4010    348947
4020        19
4050       196
4060        30
4099    145670
Name: count, dtype: int64

REGION
Missing: 0
Unique values: 11

Value counts:
REGION
1     20410
2     59797
3     69493
4     56511
5     81578
6     13816
7     40881
8     26070
9     34543
10    31783
11    59980
Name: count, dtype: int64

BLANK

## Step 13 — Validate Age at Listing

`INIT_AGE` represents patient age at listing.

EDA showed that pediatric patients are legitimately present in the dataset, so low ages should not automatically be treated as invalid.

This step checks age range and missingness before any modeling.

In [ ]:
# ============================================
# STEP 13 — Validate age at listing
# ============================================

print("AGE VALIDATION")

print("Missing age:",
      df_clean["INIT_AGE"].isna().sum())

print("Minimum age:",
      df_clean["INIT_AGE"].min())

print("Maximum age:",
      df_clean["INIT_AGE"].max())

print("Mean age:",
      round(df_clean["INIT_AGE"].mean(), 2))

print("Median age:",
      df_clean["INIT_AGE"].median())


# Check clearly impossible ages
invalid_age_mask = (
    (df_clean["INIT_AGE"] < 0) |
    (df_clean["INIT_AGE"] > 120)
)

print("\nImpossible ages (<0 or >120):",
      invalid_age_mask.sum())


# Useful age-group checks
print("\nAGE GROUP COUNTS")

print(
    "Pediatric (<18):",
    (df_clean["INIT_AGE"] < 18).sum()
)

print(
    "Adults (18-64):",
    (
        (df_clean["INIT_AGE"] >= 18) &
        (df_clean["INIT_AGE"] <= 64)
    ).sum()
)

print(
    "Older adults (65+):",
    (df_clean["INIT_AGE"] >= 65).sum()
)

print(
    "Age 80+:",
    (df_clean["INIT_AGE"] >= 80).sum()
)

AGE VALIDATION
Missing age: 0
Minimum age: 0
Maximum age: 91
Mean age: 51.87
Median age: 54.0

Impossible ages (<0 or >120): 0

AGE GROUP COUNTS
Pediatric (<18): 12622
Adults (18-64): 378562
Older adults (65+): 103678
Age 80+: 729


## Step 14 — Baseline Feature Missingness Summary

After cleaning and validation, the modeling-safe baseline features are reviewed for remaining missing values.

This step helps separate:

- complete features
- ordinary missing values
- structured missingness
- variables that will require training-only imputation later

No rows are removed and no values are imputed in this step.

In [ ]:
# ============================================
# STEP 14 — Safe feature missingness summary
# ============================================

safe_feature_missingness = []

for col in safe_listing_features:

    missing_count = df_clean[col].isna().sum()
    missing_percent = missing_count / len(df_clean) * 100

    safe_feature_missingness.append({
        "Feature": col,
        "Missing_Count": missing_count,
        "Missing_Percent": round(missing_percent, 2)
    })


safe_feature_missingness = pd.DataFrame(
    safe_feature_missingness
).sort_values(
    "Missing_Percent",
    ascending=False
)

print("SAFE FEATURE MISSINGNESS")
print(
    safe_feature_missingness
    .to_string(index=False)
)


# Check rows complete across all safe features
complete_safe_rows = (
    df_clean[safe_listing_features]
    .notna()
    .all(axis=1)
)

print("\nCOMPLETE BASELINE ROWS")
print(
    "Complete rows:",
    complete_safe_rows.sum()
)

print(
    "Complete percent:",
    round(complete_safe_rows.mean() * 100, 2),
    "%"
)

print(
    "Rows with at least one missing safe feature:",
    (~complete_safe_rows).sum()
)

SAFE FEATURE MISSINGNESS
            Feature  Missing_Count  Missing_Percent
          INIT_CPRA         142741            28.84
FUNC_STAT_TCR_clean           3976             0.80
      BMI_TCR_clean           2061             0.42
           INIT_AGE              0             0.00
                ABO              0             0.00
             ETHCAT              0             0.00
             GENDER              0             0.00
  INIT_CPRA_missing              0             0.00
        ON_DIALYSIS              0             0.00
          INIT_STAT              0             0.00
             REGION              0             0.00
       A2A2B_status              0             0.00

COMPLETE BASELINE ROWS
Complete rows: 349997
Complete percent: 70.73 %
Rows with at least one missing safe feature: 144865


## Step 15 — Missing-Value Strategy

Missing values are not imputed yet.

The final model pipeline will learn imputation values from the training data only.

Planned strategy:

- `INIT_CPRA`: median imputation + missingness indicator
- `BMI_TCR_clean`: median imputation + missingness indicator
- `FUNC_STAT_TCR_clean`: preserve as a categorical feature and represent missing values as a separate category
- structural missingness such as dialysis date and A2/A2B applicability is not blindly imputed

This prevents unnecessary row deletion and reduces preprocessing leakage.

In [ ]:
# ============================================
# STEP 15 — Create missingness indicators
# ============================================

# CPRA flag already exists from Step 7
# INIT_CPRA_missing

# BMI missingness indicator
df_clean["BMI_missing"] = (
    df_clean["BMI_TCR_clean"].isna()
).astype(int)

# Functional-status missingness indicator
df_clean["FUNC_STAT_missing"] = (
    df_clean["FUNC_STAT_TCR_clean"].isna()
).astype(int)


print("MISSINGNESS INDICATORS")

print(
    "INIT_CPRA missing:",
    df_clean["INIT_CPRA_missing"].sum()
)

print(
    "BMI missing:",
    df_clean["BMI_missing"].sum()
)

print(
    "Functional status missing:",
    df_clean["FUNC_STAT_missing"].sum()
)


print("\nPERCENT MISSING")

print(
    "INIT_CPRA:",
    round(df_clean["INIT_CPRA_missing"].mean() * 100, 2),
    "%"
)

print(
    "BMI:",
    round(df_clean["BMI_missing"].mean() * 100, 2),
    "%"
)

print(
    "Functional status:",
    round(df_clean["FUNC_STAT_missing"].mean() * 100, 2),
    "%"
)

MISSINGNESS INDICATORS
INIT_CPRA missing: 142741
BMI missing: 2061
Functional status missing: 3976

PERCENT MISSING
INIT_CPRA: 28.84 %
BMI: 0.42 %
Functional status: 0.8 %


## Step 16 — Patient and Registration ID Validation

One patient may have multiple waitlist registrations.

`PT_CODE` identifies the patient, while `WL_ID_CODE` identifies an individual waitlist registration.

Repeated patient IDs are preserved because they represent real multiple registrations.

Later, train/test splitting must keep all registrations from the same patient in the same split to prevent patient leakage.

In [ ]:
# ============================================
# STEP 16 — Validate patient and registration IDs
# ============================================

print("ID VALIDATION")

print("Total rows:",
      len(df_clean))

print("Unique WL_ID_CODE:",
      df_clean["WL_ID_CODE"].nunique())

print("Duplicate WL_ID_CODE:",
      df_clean["WL_ID_CODE"].duplicated().sum())

print("\nUnique patients (PT_CODE):",
      df_clean["PT_CODE"].nunique())


# Registrations per patient
registrations_per_patient = (
    df_clean.groupby("PT_CODE")
    .size()
)

print("\nPATIENT REGISTRATION COUNTS")

print(
    "Patients with 1 registration:",
    (registrations_per_patient == 1).sum()
)

print(
    "Patients with 2+ registrations:",
    (registrations_per_patient >= 2).sum()
)

print(
    "Patients with 3+ registrations:",
    (registrations_per_patient >= 3).sum()
)

print(
    "Maximum registrations for one patient:",
    registrations_per_patient.max()
)


# Number of rows belonging to repeated patients
repeated_patient_ids = registrations_per_patient[
    registrations_per_patient > 1
].index

repeated_patient_rows = (
    df_clean["PT_CODE"]
    .isin(repeated_patient_ids)
    .sum()
)

print(
    "\nRows belonging to patients with multiple registrations:",
    repeated_patient_rows
)

ID VALIDATION
Total rows: 494862
Unique WL_ID_CODE: 494862
Duplicate WL_ID_CODE: 0

Unique patients (PT_CODE): 414187

PATIENT REGISTRATION COUNTS
Patients with 1 registration: 349609
Patients with 2+ registrations: 64578
Patients with 3+ registrations: 12220
Maximum registrations for one patient: 11

Rows belonging to patients with multiple registrations: 145253


## Step 17 — Validate Outcome and Derived Targets

The derived modeling targets are checked against the recorded final outcome.

Expected definitions:

- `event_adverse = 1` for death or removal because the patient became too sick
- `event_transplant = 1` for transplant
- `censored = 1` for candidates who are still waiting

No target values are changed unless inconsistencies are found.

In [ ]:
# ============================================
# STEP 17 — Validate outcome-derived targets
# ============================================

print("OUTCOME COUNTS")
print(df_clean["outcome"].value_counts(dropna=False))


# Expected adverse-event target
expected_adverse = (
    df_clean["outcome"]
    .isin(["died", "removed_too_sick"])
    .astype(int)
)

adverse_mismatch = (
    df_clean["event_adverse"] != expected_adverse
)

print("\nADVERSE TARGET CHECK")
print("Mismatches:", adverse_mismatch.sum())


# Expected transplant target
expected_transplant = (
    df_clean["outcome"]
    .eq("transplanted")
    .astype(int)
)

transplant_mismatch = (
    df_clean["event_transplant"] != expected_transplant
)

print("\nTRANSPLANT TARGET CHECK")
print("Mismatches:", transplant_mismatch.sum())


# Expected censoring flag
expected_censored = (
    df_clean["outcome"]
    .eq("still_waiting")
    .astype(int)
)

censor_mismatch = (
    df_clean["censored"] != expected_censored
)

print("\nCENSORING CHECK")
print("Mismatches:", censor_mismatch.sum())


# Overall target validation
target_problem = (
    adverse_mismatch |
    transplant_mismatch |
    censor_mismatch
)

print("\nOVERALL TARGET VALIDATION")
print(
    "Rows with any target inconsistency:",
    target_problem.sum()
)

print(
    "Rows with consistent targets:",
    (~target_problem).sum()
)

OUTCOME COUNTS
outcome
transplanted              234429
still_waiting             103300
removed_administrative     45332
transplanted_elsewhere     37243
removed_too_sick           35219
died                       34524
unknown                     4815
Name: count, dtype: int64

ADVERSE TARGET CHECK
Mismatches: 0

TRANSPLANT TARGET CHECK
Mismatches: 0

CENSORING CHECK
Mismatches: 0

OVERALL TARGET VALIDATION
Rows with any target inconsistency: 0
Rows with consistent targets: 494862


## Step 18 — Flag Records Valid for Survival Analysis

Records with `END_DATE < INIT_DATE` have impossible negative follow-up time.

These records are preserved in the master cleaned dataset because they may still be usable for non-time-based analysis.

A `survival_valid` flag is created so that invalid-duration records can be excluded specifically from survival modeling.

In [ ]:
# ============================================
# STEP 18 — Create survival validity flag
# ============================================

df_clean["survival_valid"] = (
    ~df_clean["invalid_date_order"]
).astype(int)


print("SURVIVAL VALIDITY")

print(
    "Valid for survival analysis:",
    df_clean["survival_valid"].sum()
)

print(
    "Invalid for survival analysis:",
    (df_clean["survival_valid"] == 0).sum()
)


print("\nOUTCOMES OF INVALID SURVIVAL RECORDS")

print(
    df_clean.loc[
        df_clean["survival_valid"] == 0,
        "outcome"
    ].value_counts()
)


print("\nDAYS_TO_EVENT MISSING AMONG INVALID RECORDS")

print(
    df_clean.loc[
        df_clean["survival_valid"] == 0,
        "days_to_event"
    ].isna().sum()
)

SURVIVAL VALIDITY
Valid for survival analysis: 494803
Invalid for survival analysis: 59

OUTCOMES OF INVALID SURVIVAL RECORDS
outcome
died                      58
removed_administrative     1
Name: count, dtype: int64

DAYS_TO_EVENT MISSING AMONG INVALID RECORDS
59


## Step 19 — Validate Survival Duration

The existing `days_to_event` variable is compared against an independently calculated duration:

`END_DATE - INIT_DATE`

This verifies that the survival-time variable is internally consistent before it is used for modeling.

No duration values are changed yet.

In [ ]:
# ============================================
# STEP 19 — Validate survival duration
# ============================================

# Compare only rows with valid timelines
valid_time_mask = (
    (df_clean["survival_valid"] == 1) &
    df_clean["days_to_event"].notna() &
    df_clean["calculated_days_to_end"].notna()
)

duration_difference = (
    df_clean.loc[valid_time_mask, "days_to_event"]
    - df_clean.loc[valid_time_mask, "calculated_days_to_end"]
)

print("SURVIVAL DURATION CHECK")

print(
    "Rows compared:",
    valid_time_mask.sum()
)

print(
    "Exact matches:",
    (duration_difference == 0).sum()
)

print(
    "Mismatches:",
    (duration_difference != 0).sum()
)


# Show mismatched records only
duration_mismatch_mask = (
    valid_time_mask &
    (
        df_clean["days_to_event"]
        != df_clean["calculated_days_to_end"]
    )
)

duration_mismatches = df_clean.loc[
    duration_mismatch_mask,
    [
        "WL_ID_CODE",
        "outcome",
        "INIT_DATE",
        "END_DATE",
        "days_to_event",
        "calculated_days_to_end",
        "DAYSWAIT_CHRON"
    ]
]

print("\nMISMATCHED RECORDS")
print(duration_mismatches.to_string(index=False))

SURVIVAL DURATION CHECK
Rows compared: 494803
Exact matches: 494803
Mismatches: 0

MISMATCHED RECORDS
Empty DataFrame
Columns: [WL_ID_CODE, outcome, INIT_DATE, END_DATE, days_to_event, calculated_days_to_end, DAYSWAIT_CHRON]
Index: []


## Step 20 — Compare Source Wait Time with Derived Survival Time

`days_to_event` exactly matches the duration calculated from `END_DATE - INIT_DATE`.

This step compares the derived duration with the source field `DAYSWAIT_CHRON` to identify the small number of remaining inconsistencies.

No values are changed yet.

In [ ]:
# ============================================
# STEP 20 — Compare DAYSWAIT_CHRON
# with days_to_event
# ============================================

wait_compare_mask = (
    df_clean["survival_valid"].eq(1) &
    df_clean["days_to_event"].notna() &
    df_clean["DAYSWAIT_CHRON"].notna()
)

wait_difference = (
    df_clean.loc[wait_compare_mask, "DAYSWAIT_CHRON"]
    - df_clean.loc[wait_compare_mask, "days_to_event"]
)

print("WAIT-TIME CONSISTENCY CHECK")

print(
    "Rows compared:",
    wait_compare_mask.sum()
)

print(
    "Exact matches:",
    (wait_difference == 0).sum()
)

print(
    "Mismatches:",
    (wait_difference != 0).sum()
)


wait_mismatch_mask = (
    wait_compare_mask &
    (
        df_clean["DAYSWAIT_CHRON"]
        != df_clean["days_to_event"]
    )
)

wait_mismatches = df_clean.loc[
    wait_mismatch_mask,
    [
        "WL_ID_CODE",
        "outcome",
        "INIT_DATE",
        "END_DATE",
        "days_to_event",
        "DAYSWAIT_CHRON",
        "calculated_days_to_end"
    ]
]

print("\nMISMATCHED RECORDS")
print(wait_mismatches.to_string(index=False))

WAIT-TIME CONSISTENCY CHECK
Rows compared: 494803
Exact matches: 494798
Mismatches: 5

MISMATCHED RECORDS
 WL_ID_CODE      outcome  INIT_DATE   END_DATE  days_to_event  DAYSWAIT_CHRON  calculated_days_to_end
    1949035 transplanted 2025-12-19 2026-06-25          188.0            45.0                     188
    1959016 transplanted 2026-02-13 2026-04-07           53.0             2.0                      53
    1943683 transplanted 2025-11-26 2026-05-12          167.0             0.0                     167
    1637077 transplanted 2021-12-08 2022-02-10           64.0            36.0                      64
    1606052 transplanted 2021-06-22 2021-08-25           64.0            37.0                      64


## Step 21 — Create Final Survival-Time Variable

The derived duration from `END_DATE - INIT_DATE` matches `days_to_event` for all valid records.

Five records disagree with the source field `DAYSWAIT_CHRON`.

Therefore, the cleaned survival-time variable is based on the validated date-derived duration.

Records with impossible negative timelines remain missing for survival modeling.

In [ ]:
# ============================================
# STEP 21 — Create final cleaned survival time
# ============================================

# Final validated survival duration
df_clean["survival_time_days_clean"] = np.where(
    df_clean["survival_valid"] == 1,
    df_clean["calculated_days_to_end"],
    np.nan
)

# Flag rows where source wait time disagrees
df_clean["DAYSWAIT_CHRON_mismatch"] = (
    df_clean["survival_valid"].eq(1) &
    df_clean["DAYSWAIT_CHRON"].notna() &
    (
        df_clean["DAYSWAIT_CHRON"]
        != df_clean["survival_time_days_clean"]
    )
).astype(int)


print("FINAL SURVIVAL TIME CHECK")

print(
    "Valid survival times:",
    df_clean["survival_time_days_clean"].notna().sum()
)

print(
    "Missing survival times:",
    df_clean["survival_time_days_clean"].isna().sum()
)

print(
    "Minimum valid time:",
    df_clean["survival_time_days_clean"].min()
)

print(
    "Maximum valid time:",
    df_clean["survival_time_days_clean"].max()
)

print(
    "\nDAYSWAIT_CHRON mismatches:",
    df_clean["DAYSWAIT_CHRON_mismatch"].sum()
)


# Confirm final survival time matches days_to_event
comparison_mask = (
    df_clean["survival_time_days_clean"].notna() &
    df_clean["days_to_event"].notna()
)

print(
    "Final survival time vs days_to_event mismatches:",
    (
        df_clean.loc[
            comparison_mask,
            "survival_time_days_clean"
        ]
        !=
        df_clean.loc[
            comparison_mask,
            "days_to_event"
        ]
    ).sum()
)

FINAL SURVIVAL TIME CHECK
Valid survival times: 494803
Missing survival times: 59
Minimum valid time: 0.0
Maximum valid time: 4200.0

DAYSWAIT_CHRON mismatches: 5
Final survival time vs days_to_event mismatches: 0


## Step 22 — Audit Unassigned Source Columns

Before creating the final modeling dataset, every original source variable is reviewed.

Columns that have not yet been classified as:

- safe baseline predictors
- identifiers
- targets
- future/leakage variables

are identified for additional review.

No columns are dropped in this step.

In [ ]:
# ============================================
# STEP 22 — Find unassigned original columns
# ============================================

# Original source columns only
original_columns = list(df_raw.columns)

# Columns we have already assigned a clear role
assigned_original_columns = set(
    [
        # Original safe variables
        "INIT_AGE",
        "GENDER",
        "ETHCAT",
        "ABO",
        "INIT_CPRA",
        "ON_DIALYSIS",
        "BMI_TCR",
        "FUNC_STAT_TCR",
        "INIT_STAT",
        "REGION",
        "A2A2B_ELIGIBILITY",

        # IDs
        "PT_CODE",
        "WL_ID_CODE",
        "TRR_ID_CODE",
        "DONOR_ID",

        # Targets
        "outcome",
        "event_adverse",
        "event_transplant",
        "censored",
        "days_to_event",

        # Known future / leakage
        "END_DATE",
        "END_CPRA",
        "END_STAT",
        "REM_CD",
        "TX_DATE",
        "COMPOSITE_DEATH_DATE",
        "DAYSWAIT_CHRON",

        # Listing date
        "INIT_DATE"
    ]
)

unassigned_columns = [
    col for col in original_columns
    if col not in assigned_original_columns
]

print("UNASSIGNED ORIGINAL COLUMNS")
print("Count:", len(unassigned_columns))

for col in unassigned_columns:
    print("•", col)


print("\nMISSINGNESS + UNIQUE VALUES")

unassigned_summary = []

for col in unassigned_columns:

    unassigned_summary.append({
        "Column": col,
        "Missing_Count": df_clean[col].isna().sum(),
        "Missing_Percent": round(
            df_clean[col].isna().mean() * 100, 2
        ),
        "Unique_Values": df_clean[col].nunique(dropna=True)
    })

unassigned_summary = pd.DataFrame(unassigned_summary)

print(
    unassigned_summary
    .sort_values("Missing_Percent", ascending=False)
    .to_string(index=False)
)

UNASSIGNED ORIGINAL COLUMNS
Count: 10
• DIALYSIS_DATE
• DAYSWAIT_ALLOC
• PREV_TX
• DON_TY
• DIAG_KI
• MULTIORG
• ORGAN
• PSTATUS
• PTIME
• LISTING_CTR_CODE

MISSINGNESS + UNIQUE VALUES
          Column  Missing_Count  Missing_Percent  Unique_Values
        MULTIORG         483004            97.60              1
           PTIME         267350            54.03           3982
         DIAG_KI         265540            53.66             69
           ORGAN         264107            53.37              2
         PREV_TX         264107            53.37              2
          DON_TY         264107            53.37              3
         PSTATUS         264107            53.37              2
   DIALYSIS_DATE         134245            27.13           8953
  DAYSWAIT_ALLOC           6745             1.36           7379
LISTING_CTR_CODE              0             0.00            256


## Step 23 — Assign Roles to Remaining Variables

The remaining source variables are classified before building the modeling dataset.

Some variables are clearly post-transplant or post-outcome and are excluded from baseline prediction.

Other variables have uncertain timing or conditional meaning and are retained for review rather than being used automatically.

Listing center is preserved for system-level and fairness analysis but excluded from the initial baseline model.

In [ ]:
# ============================================
# STEP 23 — Assign roles to remaining columns
# ============================================

remaining_column_roles = pd.DataFrame([
    {
        "Column": "DIALYSIS_DATE",
        "Role": "REVIEW",
        "Reason": "May support pre-listing dialysis-duration feature, but timing inconsistencies require caution."
    },
    {
        "Column": "DAYSWAIT_ALLOC",
        "Role": "REVIEW",
        "Reason": "Allocation-time measure; exact timing and leakage risk should be confirmed."
    },
    {
        "Column": "PREV_TX",
        "Role": "REVIEW",
        "Reason": "Previous-transplant field, but timing/availability in this extract is not fully confirmed."
    },
    {
        "Column": "DON_TY",
        "Role": "EXCLUDE_BASELINE",
        "Reason": "Donor information is known at transplant, not at listing."
    },
    {
        "Column": "DIAG_KI",
        "Role": "REVIEW",
        "Reason": "Diagnosis field is highly missing and timing in this extract requires confirmation."
    },
    {
        "Column": "MULTIORG",
        "Role": "REVIEW",
        "Reason": "97.6% missing and appears conditionally populated."
    },
    {
        "Column": "ORGAN",
        "Role": "EXCLUDE_BASELINE",
        "Reason": "Transplant-related organ information is post-listing."
    },
    {
        "Column": "PSTATUS",
        "Role": "EXCLUDE_BASELINE",
        "Reason": "Patient-status field is associated with transplant/post-listing information."
    },
    {
        "Column": "PTIME",
        "Role": "EXCLUDE_BASELINE",
        "Reason": "Time-related post-transplant/post-outcome variable."
    },
    {
        "Column": "LISTING_CTR_CODE",
        "Role": "SYSTEM_ANALYSIS",
        "Reason": "Known center identifier; preserve for fairness/system analysis but exclude from first baseline model."
    }
])

print(remaining_column_roles.to_string(index=False))

          Column             Role                                                                                                Reason
   DIALYSIS_DATE           REVIEW        May support pre-listing dialysis-duration feature, but timing inconsistencies require caution.
  DAYSWAIT_ALLOC           REVIEW                           Allocation-time measure; exact timing and leakage risk should be confirmed.
         PREV_TX           REVIEW            Previous-transplant field, but timing/availability in this extract is not fully confirmed.
          DON_TY EXCLUDE_BASELINE                                             Donor information is known at transplant, not at listing.
         DIAG_KI           REVIEW                   Diagnosis field is highly missing and timing in this extract requires confirmation.
        MULTIORG           REVIEW                                                    97.6% missing and appears conditionally populated.
           ORGAN EXCLUDE_BASELINE               

## Step 24 — Build the Clean Baseline Modeling Dataset

A separate baseline modeling dataframe is created from the cleaned master dataframe.

It contains only:

- identifiers needed for tracking and patient-grouped splitting
- approved listing-time predictors
- missingness indicators
- classification targets
- survival-analysis fields
- validation metadata

Post-outcome and leakage variables are excluded from this baseline table.

No imputation or categorical encoding is performed yet.

In [ ]:
# ============================================
# STEP 24 — Build baseline modeling dataset
# ============================================

# Final baseline predictors
baseline_features = [
    "INIT_AGE",
    "GENDER",
    "ETHCAT",
    "ABO",
    "INIT_CPRA",
    "INIT_CPRA_missing",
    "ON_DIALYSIS",
    "BMI_TCR_clean",
    "BMI_missing",
    "FUNC_STAT_TCR_clean",
    "FUNC_STAT_missing",
    "INIT_STAT",
    "REGION",
    "A2A2B_status"
]

# Tracking / split columns
tracking_columns = [
    "PT_CODE",
    "WL_ID_CODE"
]

# Validation metadata
metadata_columns = [
    "INIT_DATE",
    "listing_year",
    "LISTING_CTR_CODE"
]

# Targets and survival information
outcome_columns = [
    "outcome",
    "event_adverse",
    "event_transplant",
    "censored",
    "survival_valid",
    "survival_time_days_clean"
]

baseline_columns = (
    tracking_columns
    + metadata_columns
    + baseline_features
    + outcome_columns
)

modeling_base_df = df_clean[
    baseline_columns
].copy()


print("BASELINE MODELING DATASET CREATED")

print("Shape:", modeling_base_df.shape)
print("Rows:", len(modeling_base_df))
print("Columns:", modeling_base_df.shape[1])


print("\nBASELINE FEATURES")
for col in baseline_features:
    print("✓", col)


print("\nALL MODELING TABLE COLUMNS")
print(modeling_base_df.columns.tolist())


print("\nDUPLICATE REGISTRATION IDs")
print(
    modeling_base_df["WL_ID_CODE"]
    .duplicated()
    .sum()
)


print("\nTARGET MISSINGNESS")
print(
    modeling_base_df[
        [
            "event_adverse",
            "event_transplant",
            "censored"
        ]
    ]
    .isna()
    .sum()
)

BASELINE MODELING DATASET CREATED
Shape: (494862, 25)
Rows: 494862
Columns: 25

BASELINE FEATURES
✓ INIT_AGE
✓ GENDER
✓ ETHCAT
✓ ABO
✓ INIT_CPRA
✓ INIT_CPRA_missing
✓ ON_DIALYSIS
✓ BMI_TCR_clean
✓ BMI_missing
✓ FUNC_STAT_TCR_clean
✓ FUNC_STAT_missing
✓ INIT_STAT
✓ REGION
✓ A2A2B_status

ALL MODELING TABLE COLUMNS
['PT_CODE', 'WL_ID_CODE', 'INIT_DATE', 'listing_year', 'LISTING_CTR_CODE', 'INIT_AGE', 'GENDER', 'ETHCAT', 'ABO', 'INIT_CPRA', 'INIT_CPRA_missing', 'ON_DIALYSIS', 'BMI_TCR_clean', 'BMI_missing', 'FUNC_STAT_TCR_clean', 'FUNC_STAT_missing', 'INIT_STAT', 'REGION', 'A2A2B_status', 'outcome', 'event_adverse', 'event_transplant', 'censored', 'survival_valid', 'survival_time_days_clean']

DUPLICATE REGISTRATION IDs
0

TARGET MISSINGNESS
event_adverse       0
event_transplant    0
censored            0
dtype: int64


## Step 25 — Final Data-Cleaning Quality Audit

The completed cleaning pipeline is validated before the cleaned datasets are saved.

This audit confirms that:

- the raw source dataframe remains unchanged
- no registrations were unintentionally removed
- cleaned numerical variables satisfy expected quality checks
- identifiers and targets remain consistent
- invalid survival timelines remain explicitly flagged
- missingness is preserved where appropriate

In [ ]:
# ============================================
# STEP 25 — Final cleaning quality audit
# ============================================

print("=" * 60)
print("FINAL DATA CLEANING AUDIT")
print("=" * 60)

# 1. Dataset size
print("\n1. DATASET SIZE")
print("Raw rows:", len(df_raw))
print("Clean rows:", len(df_clean))
print(
    "Rows accidentally removed:",
    len(df_raw) - len(df_clean)
)


# 2. Registration IDs
print("\n2. REGISTRATION IDs")
print(
    "Duplicate WL_ID_CODE:",
    df_clean["WL_ID_CODE"].duplicated().sum()
)

print(
    "Unique patients:",
    df_clean["PT_CODE"].nunique()
)


# 3. BMI
print("\n3. CLEANED BMI")
print(
    "Minimum cleaned BMI:",
    df_clean["BMI_TCR_clean"].min()
)

print(
    "Maximum cleaned BMI:",
    df_clean["BMI_TCR_clean"].max()
)

print(
    "Missing cleaned BMI:",
    df_clean["BMI_TCR_clean"].isna().sum()
)


# 4. Functional status
print("\n4. FUNCTIONAL STATUS")
print(
    "Remaining FUNC_STAT = 1:",
    (df_clean["FUNC_STAT_TCR_clean"] == 1).sum()
)

print(
    "Missing cleaned functional status:",
    df_clean["FUNC_STAT_TCR_clean"].isna().sum()
)


# 5. Initial CPRA
print("\n5. INITIAL CPRA")
print(
    "Values outside 0-100:",
    (
        (df_clean["INIT_CPRA"] < 0) |
        (df_clean["INIT_CPRA"] > 100)
    ).sum()
)

print(
    "Missing INIT_CPRA:",
    df_clean["INIT_CPRA"].isna().sum()
)


# 6. Targets
print("\n6. TARGETS")
print(
    "Missing event_adverse:",
    df_clean["event_adverse"].isna().sum()
)

print(
    "Missing event_transplant:",
    df_clean["event_transplant"].isna().sum()
)

print(
    "Missing censored:",
    df_clean["censored"].isna().sum()
)


# 7. Survival data
print("\n7. SURVIVAL DATA")
print(
    "Valid survival records:",
    df_clean["survival_valid"].sum()
)

print(
    "Invalid survival records:",
    (df_clean["survival_valid"] == 0).sum()
)

print(
    "Missing cleaned survival time:",
    df_clean["survival_time_days_clean"].isna().sum()
)

print(
    "Negative cleaned survival times:",
    (df_clean["survival_time_days_clean"] < 0).sum()
)


# 8. Modeling table
print("\n8. BASELINE MODELING TABLE")
print(
    "Shape:",
    modeling_base_df.shape
)

print(
    "Number of baseline features:",
    len(baseline_features)
)


print("\n" + "=" * 60)
print("AUDIT COMPLETE")
print("=" * 60)

FINAL DATA CLEANING AUDIT

1. DATASET SIZE
Raw rows: 494862
Clean rows: 494862
Rows accidentally removed: 0

2. REGISTRATION IDs
Duplicate WL_ID_CODE: 0
Unique patients: 414187

3. CLEANED BMI
Minimum cleaned BMI: 10.03
Maximum cleaned BMI: 97.65
Missing cleaned BMI: 2061

4. FUNCTIONAL STATUS
Remaining FUNC_STAT = 1: 0
Missing cleaned functional status: 3976

5. INITIAL CPRA
Values outside 0-100: 0
Missing INIT_CPRA: 142741

6. TARGETS
Missing event_adverse: 0
Missing event_transplant: 0
Missing censored: 0

7. SURVIVAL DATA
Valid survival records: 494803
Invalid survival records: 59
Missing cleaned survival time: 59
Negative cleaned survival times: 0

8. BASELINE MODELING TABLE
Shape: (494862, 25)
Number of baseline features: 14

AUDIT COMPLETE


## Step 26 — Save Cleaned Outputs

The completed cleaning pipeline produces:

1. A cleaned master dataset containing the original variables and cleaning flags.
2. A baseline modeling dataset containing approved features, identifiers, targets, and validation metadata.
3. A cleaning log documenting the major preprocessing decisions.

Row-level datasets should remain in approved/private storage and should not be committed to a public repository.

In [ ]:
# ============================================
# STEP 26 — Save cleaned outputs
# ============================================

cleaning_log = pd.DataFrame([
    {
        "Area": "Raw data",
        "Decision": "Preserved original dataframe",
        "Details": "All cleaning performed on df_clean; df_raw kept unchanged."
    },
    {
        "Area": "Dates",
        "Decision": "Converted date fields to datetime",
        "Details": "INIT_DATE, END_DATE, DIALYSIS_DATE, TX_DATE, and COMPOSITE_DEATH_DATE converted."
    },
    {
        "Area": "Invalid timelines",
        "Decision": "Flagged rather than deleted",
        "Details": "59 rows have END_DATE before INIT_DATE and are excluded only from survival-duration modeling."
    },
    {
        "Area": "BMI",
        "Decision": "Created BMI_TCR_clean",
        "Details": "BMI <= 10 or > 100 converted to missing. Original BMI preserved."
    },
    {
        "Area": "Functional status",
        "Decision": "Created FUNC_STAT_TCR_clean",
        "Details": "Single unsupported value 1 converted to missing. Codes 996 and 998 preserved."
    },
    {
        "Area": "Initial CPRA",
        "Decision": "Preserved missing values and created missingness flag",
        "Details": "No imputation performed because missingness changes sharply after 2022."
    },
    {
        "Area": "A2/A2B eligibility",
        "Decision": "Created A2A2B_status",
        "Details": "Separated non-B/not-applicable from B recorded 0, B recorded 1, and B missing."
    },
    {
        "Area": "Dialysis date",
        "Decision": "Preserved structural missingness",
        "Details": "All ON_DIALYSIS=N records lack DIALYSIS_DATE; timing inconsistencies retained for review."
    },
    {
        "Area": "Repeated patients",
        "Decision": "Preserved registrations",
        "Details": "PT_CODE retained so future train/test splits can keep the same patient in one split."
    },
    {
        "Area": "Target validation",
        "Decision": "No corrections required",
        "Details": "event_adverse, event_transplant, and censored matched outcome for all 494,862 rows."
    },
    {
        "Area": "Survival time",
        "Decision": "Created survival_time_days_clean",
        "Details": "Based on END_DATE - INIT_DATE for 494,803 valid timelines; 59 invalid timelines remain missing."
    },
    {
        "Area": "Leakage",
        "Decision": "Excluded future information from baseline features",
        "Details": "End status/date, transplant fields, outcome-related fields, and wait duration are not baseline predictors."
    }
])


# Save compressed row-level files locally in Colab
df_clean.to_csv(
    "/content/medimate_cleaned_master.csv.gz",
    index=False,
    compression="gzip"
)

modeling_base_df.to_csv(
    "/content/medimate_modeling_base.csv.gz",
    index=False,
    compression="gzip"
)

cleaning_log.to_csv(
    "/content/medimate_cleaning_log.csv",
    index=False
)


print("FILES SAVED")

print("/content/medimate_cleaned_master.csv.gz")
print("/content/medimate_modeling_base.csv.gz")
print("/content/medimate_cleaning_log.csv")

print("\nCleaning log:")
print(cleaning_log.to_string(index=False))

FILES SAVED
/content/medimate_cleaned_master.csv.gz
/content/medimate_modeling_base.csv.gz
/content/medimate_cleaning_log.csv

Cleaning log:
              Area                                              Decision                                                                                                    Details
          Raw data                          Preserved original dataframe                                                 All cleaning performed on df_clean; df_raw kept unchanged.
             Dates                     Converted date fields to datetime                           INIT_DATE, END_DATE, DIALYSIS_DATE, TX_DATE, and COMPOSITE_DEATH_DATE converted.
 Invalid timelines                           Flagged rather than deleted              59 rows have END_DATE before INIT_DATE and are excluded only from survival-duration modeling.
               BMI                                 Created BMI_TCR_clean                                           BMI <= 10 or > 100 conve